# Lab 01-03 · Derivatives

Companion to chapter [01-03](../../chapters/01-03-derivatives.md). Handout: [README.md](README.md).

Run the cells in order. Cells marked **🔮 Predict** ask you to write a prediction *before* running.

In [ ]:
import math, shutil, subprocess, sys
from pathlib import Path
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
sys.path.insert(0, str(Path.cwd().parents[3] / "tools"))
try:
    from plotstyle import use_style; use_style()
except ImportError:
    pass

## Part 1 — From secants to the tangent, and the rules in SymPy

In [ ]:
f = lambda x: x**2
for h in [1, 0.1, 0.01, 0.001]:
    print(f"h = {h:<6} secant slope at x = 2: {(f(2 + h) - f(2)) / h:.4f}")

x, t, w, tau = sp.symbols("x t omega tau", positive=True)
checks = {
    "product":  sp.diff(x**2 * sp.exp(x), x),
    "quotient": sp.simplify(sp.diff(sp.sin(x) / x, x)),
    "chain":    sp.diff(sp.exp(-t / tau), t),
    "sinusoid": sp.diff(sp.cos(w * t), t),
    "arctan":   sp.diff(sp.atan(x), x),
    "ln":       sp.diff(sp.log(x), x),
}
for k, v in checks.items():
    print(f"{k:<9} {v}")
assert sp.simplify(checks["arctan"] - 1 / (1 + x**2)) == 0

## Part 2 — Numerical derivatives: forward versus central

**🔮 Predict:** at what step $h$ is each method most accurate, and how many correct digits does each reach?

In [ ]:
hs = np.logspace(-15, -1, 300)
x0 = 1.0
fwd = np.abs((np.sin(x0 + hs) - np.sin(x0)) / hs - np.cos(x0))
cen = np.abs((np.sin(x0 + hs) - np.sin(x0 - hs)) / (2 * hs) - np.cos(x0))
plt.figure(figsize=(6.5, 4))
plt.loglog(hs, np.maximum(fwd, 1e-17), ".", ms=3, label="forward")
plt.loglog(hs, np.maximum(cen, 1e-17), ".", ms=3, label="central")
plt.xlabel("h"); plt.ylabel("|error|"); plt.legend(); plt.show()
print(f"forward best: h = {hs[np.argmin(fwd)]:.1e}, error {fwd.min():.1e}")
print(f"central best: h = {hs[np.argmin(cen)]:.1e}, error {cen.min():.1e}")
assert cen.min() < 1e-9 and fwd.min() > cen.min()

## Part 3 — Capacitor and inductor waveforms from sampled data

Differentiate the trapezoidal voltage of Problem 01-03.7 numerically (`np.gradient` uses central differences) and multiply by $C$.

In [ ]:
C = 10e-6
tt = np.linspace(0, 8e-3, 8001)
v = np.interp(tt, [0, 2e-3, 5e-3, 6e-3, 8e-3], [0, 10, 10, 0, 0])
i = C * np.gradient(v, tt)
fig, ax = plt.subplots(2, 1, figsize=(7, 4.5), sharex=True)
ax[0].plot(tt * 1e3, v); ax[0].set_ylabel("v (V)")
ax[1].plot(tt * 1e3, i * 1e3, color="C1"); ax[1].set_ylabel("i (mA)"); ax[1].set_xlabel("t (ms)")
plt.show()
print(f"i at 1 ms = {np.interp(1e-3, tt, i)*1e3:.2f} mA, at 5.5 ms = {np.interp(5.5e-3, tt, i)*1e3:.2f} mA")
w_peak = 0.5 * C * 10**2
print(f"stored energy at the peak = {w_peak*1e3:.2f} mJ")

**Energy bookkeeping.** Integrate $p = vi$ over time with the trapezoid rule; the running total should match $\tfrac{1}{2}Cv^2$ at every instant (the chain rule of §3.8, in numbers).

In [ ]:
p = v * i
w_num = np.concatenate([[0], np.cumsum(0.5 * (p[1:] + p[:-1]) * np.diff(tt))])
plt.figure(figsize=(6.5, 3.4))
plt.plot(tt * 1e3, w_num * 1e3, label="∫ p dt")
plt.plot(tt * 1e3, 0.5 * C * v**2 * 1e3, "--", label="½ C v²")
plt.xlabel("t (ms)"); plt.ylabel("energy (mJ)"); plt.legend(); plt.show()
assert np.max(np.abs(w_num - 0.5 * C * v**2)) < 2e-6

## Part 4 — Linearizing the diode

**🔮 Predict:** the small-signal resistance at 1 mA and at 2 mA.

In [ ]:
VT, IS = 0.02585, 1e-14
for I0 in [1e-3, 2e-3]:
    V0 = VT * math.log(I0 / IS)
    for dV in [1e-3, 5e-3, 20e-3]:
        exact = IS * math.exp((V0 + dV) / VT) - I0
        lin = I0 / VT * dV
        print(f"I = {I0*1e3:.0f} mA, r_d = {VT/I0:6.2f} Ω, dV = {dV*1e3:4.0f} mV: "
              f"linear {lin*1e6:8.2f} µA, exact {exact*1e6:8.2f} µA, error {100*(lin/exact - 1):6.1f}%")

## Part 5 — ngspice

`cap_ind.cir` and `diode_ac.cir` are checked automatically by `python tools/run_spice.py 01-03` from the repository root. The cell below runs them directly if ngspice is installed.

In [ ]:
if shutil.which("ngspice"):
    for net in ["cap_ind.cir", "diode_ac.cir"]:
        out = subprocess.run(["ngspice", "-b", net], capture_output=True, text=True).stdout
        keys = ("ic1_ramp_up", "ic1_hold", "ic1_ramp_down", "ic2_ramp", "vl_kick", "rd_1m", "rd_2m")
        print(net)
        for l in out.splitlines():
            if "=" in l and l.split("=")[0].strip() in keys:
                print("   " + l.strip())
else:
    print("ngspice not found; see SETUP.md. Skipping.")

## Your turn

1. In Part 3, replace the trapezoid with a sine of 1 kHz and 10 V amplitude. Predict the current amplitude, then check.
2. In Part 4, at what $\Delta V$ does the linear estimate reach 10% error? Relate your answer to $V_T$.